# 03 — Review, Final Train, Inference

Review the fixed-split results, lock one Stage-2 configuration, retrain it on all labeled ROIs, then run final inference.


In [ ]:
from pathlib import Path
import os
import sys

PROJECT_DIR = Path('/media/ibmelab/ibme31/YUD/PUMA/Version 13').expanduser().resolve()
# Colab alternative: PROJECT_DIR = Path('/content/drive/MyDrive/Research/PUMA')
%cd {PROJECT_DIR}
for module_name in list(sys.modules):
    if module_name == 'puma' or module_name.startswith('puma.'):
        del sys.modules[module_name]
if str(PROJECT_DIR) in sys.path:
    sys.path.remove(str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR))

os.environ.setdefault('PUMA_LORA_GRAD_CHECKPOINTING', '1')
os.environ.setdefault('PUMA_V13_AUTO_OOM_FALLBACK', '1')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')


In [ ]:
%pip install -q -r requirements_colab.txt


In [ ]:
from puma.runtime import create_runtime, resolve_hf_token

runtime = create_runtime(
    PROJECT_DIR,
    run_folds=(0, 1, 2, 3, 4),
    seeds=(0,),
    epochs=60,
    effective_batch_size=256,
    stage2_micro_batch_size=256,
    preprocessing_workers=0,
)
runtime.training.number_of_workers = 4
runtime.paths.stage2_output_dir = PROJECT_DIR / 'PUMA_stage2_V13_outputs'
runtime.paths.ensure()
HF_TOKEN = resolve_hf_token()
print(runtime.as_dict())


In [ ]:
from puma.pipeline.experiments_v13 import aggregate_v13_results, v13_experiment_status
from puma.stage2.catalog import VERSION13_EXPERIMENTS

status = v13_experiment_status(runtime)
display(status)
ranking = aggregate_v13_results(runtime, VERSION13_EXPERIMENTS)
if ranking.empty:
    print('No complete V13 ranking is available yet.')
else:
    sort_columns = [c for c in ('macro_f1','conditional_type_macro_f1_present','reject_f1') if c in ranking.columns]
    display(ranking.sort_values(sort_columns, ascending=False, na_position='last'))


In [ ]:
CREATE_DEVELOPMENT_LOCK = False
SELECTED_EXPERIMENT = None

if CREATE_DEVELOPMENT_LOCK:
    from puma.pipeline.experiments_v13 import lock_v13_winner

    stage2_lock = lock_v13_winner(
        runtime,
        selected_experiment=SELECTED_EXPERIMENT,
        candidate_experiments=VERSION13_EXPERIMENTS,
    )
    print('Selected:', stage2_lock['selected_experiment'])
    print('Final plan:', stage2_lock['final_training_plan'])
else:
    import json

    lock_path = runtime.paths.stage2_file('stage2_v13_lock.json')
    if lock_path.exists():
        stage2_lock = json.loads(lock_path.read_text(encoding='utf-8'))
        print('Existing lock:', stage2_lock['selected_experiment'])
    else:
        print('No development lock yet.')


In [ ]:
TRAIN_FINAL_MODEL = False
FORCE_FINAL_RETRAIN = False

if TRAIN_FINAL_MODEL:
    from puma.pipeline.final_v13 import train_final_stage2_v13, write_fixed_stage1_lock_v13

    write_fixed_stage1_lock_v13(runtime)
    final_lock = train_final_stage2_v13(
        runtime,
        hf_token=HF_TOKEN,
        force=FORCE_FINAL_RETRAIN,
        auto_oom_fallback=True,
    )
    print('Final checkpoint:', final_lock['final_checkpoint'])
    final_lock


In [ ]:
CHECK_FINAL_READY = False
if CHECK_FINAL_READY:
    from puma.pipeline.final_v13 import final_v13_ready

    print(final_v13_ready(runtime))


In [ ]:
RUN_INFERENCE = False
INFERENCE_DIR = PROJECT_DIR / 'Dataset' / 'challenge_test_images'

if RUN_INFERENCE:
    from puma.pipeline.inference import run_inference

    inference_summary = run_inference(runtime, input_dir=INFERENCE_DIR, hf_token=HF_TOKEN)
    print(inference_summary)
